# NC Sound Hub — engine verification and benchmark

Runs the JUCE-free half of the DAW on Kaggle compute: the **whole engine
correctness suite** (Catch2, 28 test files) and **`nc_bench`** (per-kernel
timings against `tools/kaggle/baseline.json`), then writes a machine-readable
evidence file that feeds the release readiness profile.

Source comes from the attached dataset `terrenceperry/ncsound-daw-src` (this repo has no git remote).
Catch2 is vendored at `third_party/catch2`, so the kernel runs **with internet
off**.

What this **cannot** measure: the JUCE app/plug-in targets and `renderBlock`
(Windows-only, need MSVC + a device stack). Those rows stay marked unverified
rather than guessed.


In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "repo"
INPUT = pathlib.Path("/kaggle/input")

# Kaggle has moved dataset mounts between /kaggle/input/<slug> and
# /kaggle/input/datasets/<owner>/<slug>, and an uploaded tree may land at the
# mount root or in a repo/ subdir. Resolve the root by looking for the repo's
# CMakeLists.txt instead of hard-coding one layout.
CANDIDATES = [
    INPUT / "datasets/terrenceperry/ncsound-daw-src",
    INPUT / "ncsound-daw-src",
    INPUT / "ncsound-daw-src/repo",
]
SRC = next((c for c in CANDIDATES if (c / "CMakeLists.txt").is_file()), None)
if SRC is None:
    hits = [p.parent for p in INPUT.rglob("CMakeLists.txt")
            if (p.parent / "engine/include").is_dir()]
    SRC = hits[0] if hits else None
assert SRC is not None, (
    "repo not found; input tree: "
    + str(sorted(str(p) for p in INPUT.rglob("*"))[:40]))
print("source root:", SRC)

if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SRC, REPO, symlinks=True)

# Fingerprint so a result can be tied to exact sources without git.
import hashlib
h = hashlib.sha256()
tracked = sorted([p for p in REPO.rglob("*") if p.is_file()
                  and ".git" not in p.parts
                  and "third_party/catch2" not in p.as_posix()])
for p in tracked:
    h.update(p.relative_to(REPO).as_posix().encode())
    h.update(str(p.stat().st_size).encode())
print("files:", len(tracked))
print("source fingerprint:", h.hexdigest()[:16])
print("catch2 vendored:", (REPO / "third_party/catch2/CMakeLists.txt").is_file())
print("juce present (must be False on Linux path):", (REPO / "third_party/JUCE").exists())


In [ ]:
# Build + gate on the full engine suite + collect the benchmark.
# EXTRA_CMAKE_ARGS points CPM at the vendored Catch2 so no download is needed.
log = WORK / "build.log"
env = dict(
    os.environ,
    OUT=str(WORK / "run.json"),
    EXTRA_CMAKE_ARGS=f"-DCPM_Catch2_SOURCE={REPO}/third_party/catch2",
    JOBS=str(os.cpu_count() or 2),
)
t0 = time.time()
with open(log, "w") as fh:
    proc = subprocess.run(["bash", "tools/kaggle/run_bench.sh"], cwd=REPO,
                          env=env, stdout=fh, stderr=subprocess.STDOUT)
BUILD_SECONDS = round(time.time() - t0, 1)
text = log.read_text(errors="replace")
print("exit", proc.returncode, "in", BUILD_SECONDS, "s")
print("\n".join(text.splitlines()[-40:]))
assert proc.returncode == 0, "build/test/bench failed - see build.log"

# Second pass with a JUnit report, so every case is attributable.
bin_ = REPO / "build-kaggle-linux" / "tests" / "nc_tests"
junit = WORK / "junit.xml"
via = None
for reporter in ("junit", "xml"):
    r = subprocess.run([str(bin_), "--reporter", reporter, "-o", str(junit)],
                       capture_output=True, text=True)
    if r.returncode == 0 and junit.is_file() and junit.stat().st_size > 0:
        via = reporter
        break
print("junit reporter:", via)


In [ ]:
import collections, traceback, xml.etree.ElementTree as ET
import pandas as pd
pd.set_option("display.width", 200)

R = {}
def step(name, fn):
    try:
        R[name] = fn()
        print("OK  ", name)
    except Exception:
        R[name] = None
        print("FAIL", name)
        traceback.print_exc()

def suite():
    root = ET.parse(junit).getroot()
    cases = []
    for tc in root.iter("testcase"):
        bad = [c for c in tc if c.tag in ("failure", "error")]
        skip = [c for c in tc if c.tag == "skipped"]
        cases.append({
            "suite": tc.get("classname") or "?",
            "name": tc.get("name") or "?",
            "status": "fail" if bad else ("skip" if skip else "pass"),
            "seconds": float(tc.get("time") or 0.0),
        })
    counts = collections.Counter(c["status"] for c in cases)
    per = collections.defaultdict(collections.Counter)
    for c in cases:
        per[c["suite"]][c["status"]] += 1
    return {"cases": cases, "counts": dict(counts), "per_suite": {k: dict(v) for k, v in per.items()}}

def bench():
    run = json.loads((WORK / "run.json").read_text())
    df = pd.DataFrame(run["kernels"])[["name", "units", "ops", "ns_per_op"]]
    base = {k["name"]: k["ns_per_op"]
            for k in json.loads((REPO / "tools/kaggle/baseline.json").read_text())["kernels"]}
    df["baseline_ns"] = df["name"].map(base)
    df["ratio"] = df["ns_per_op"] / df["baseline_ns"]
    return df.sort_values("ratio", ascending=False)

def coverage():
    eng = REPO / "engine/include/nc/engine"
    headers = sorted(p.stem for p in eng.glob("*.hpp"))
    src = "\n".join(p.read_text(errors="replace") for p in (REPO / "tests").glob("*.cpp"))
    names = {p.stem for p in (REPO / "tests").glob("*.cpp")}
    def snake(s):
        out = []
        for i, ch in enumerate(s):
            if ch.isupper() and i and not s[i-1].isupper():
                out.append("_")
            out.append(ch.lower())
        return "".join(out)
    covered = [h for h in headers
               if f"nc/engine/{h}.hpp" in src or f"test_{snake(h)}" in names]
    return {"headers": headers, "covered": covered,
            "uncovered": [h for h in headers if h not in covered]}

step("suite", suite)
step("bench", bench)
step("coverage", coverage)
print()
s = R["suite"]
if s:
    print("suite:", len(s["cases"]), "cases", s["counts"])
    slow = sorted(s["cases"], key=lambda c: -c["seconds"])[:5]
    for c in slow:
        print(f"  {c['seconds']:7.3f}s {c['status']:4} {c['suite']} :: {c['name'][:60]}")
df = R["bench"]
if df is not None:
    GATE = 1.25
    AUDIO_THREAD = ["spsc_queue.push_pop", "resample.weights_lookup",
                    "tempo_map.ppq_at_samples", "tempo_map.seconds_to_ppq",
                    "clip_timeline.clip_active_at", "stream_window.plan_and_plan_read",
                    "midi_schedule.clip_block"]
    df["on_audio_thread"] = df["name"].isin(AUDIO_THREAD)
    flags = df[df["ratio"] > GATE].copy()
    print()
    print(f"kernels measured: {len(df)}; above {GATE}x baseline: {len(flags)}")
    display(df[["name", "ns_per_op", "baseline_ns", "ratio", "on_audio_thread"]])
cov = R["coverage"]
if cov:
    print(f"engine headers {len(cov['headers'])}; referenced by the suite {len(cov['covered'])}")
    print("no direct test:", cov["uncovered"])


In [ ]:
# Evidence file: everything the readiness profile is allowed to cite.
s = R["suite"] or {"cases": [], "counts": {}, "per_suite": {}}
df = R["bench"]
GATE = 1.25
if df is not None:
    flags = df[df["ratio"] > GATE]
    bench_block = {
        "kernels_measured": int(len(df)),
        "gate": GATE,
        "within_gate": int((df["ratio"] <= GATE).sum()),
        "flagged": flags[["name", "ratio", "on_audio_thread"]].to_dict("records"),
        "missing_from_run": sorted(set(df["baseline_ns"].dropna()) - set(df["baseline_ns"].dropna())),
    }
    df.to_csv(WORK / "nc_bench.csv", index=False)
    flags.to_csv(WORK / "nc_bench_flagged.csv", index=False)
else:
    bench_block = {}

n_pass = s["counts"].get("pass", 0)
readiness = {
    "schema": "nc-kaggle-readiness/1",
    "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "source_fingerprint": h.hexdigest()[:16],
    "files": len(tracked),
    "build_seconds": BUILD_SECONDS,
    "suite": {
        "cases": len(s["cases"]),
        "pass": n_pass,
        "fail": s["counts"].get("fail", 0),
        "skip": s["counts"].get("skip", 0),
        "pass_rate": round(n_pass / max(len(s["cases"]), 1), 4),
        "by_suite": s["per_suite"],
    },
    "bench": bench_block,
    "coverage": {
        "headers": len((R["coverage"] or {}).get("headers", [])),
        "referenced": len((R["coverage"] or {}).get("covered", [])),
        "unreferenced": (R["coverage"] or {}).get("uncovered", []),
    },
    # Explicitly NOT measured by this kernel.
    "not_measured": [
        "JUCE app + plug-in targets (NCSoundHub, NCMixRack)",
        "renderBlock real-time block path (NCBenchRender, Windows only)",
        "React workspace (vitest) and typecheck",
        "installer, signing, clean-runner CI",
    ],
}
(WORK / "readiness.json").write_text(json.dumps(readiness, indent=2))
print(json.dumps({k: readiness[k] for k in ("suite", "bench", "coverage")}, indent=2)[:3000])


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if R["bench"] is not None:
    d = R["bench"].sort_values("ns_per_op")
    GATE = 1.25
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(d["name"], d["ns_per_op"],
            color=["#c0392b" if r > GATE else "#2e7d32" for r in d["ratio"]])
    ax.set_xscale("log")
    ax.set_xlabel("ns per op (log)")
    ax.set_title("nc_bench: component cost (red = above 1.25x baseline)")
    plt.tight_layout()
    fig.savefig(WORK / "nc_bench.png", dpi=120)
    print("wrote nc_bench.png")

# Drop the source copy so the downloaded artifact is the evidence, not 55 MB
# of duplicated repository.
shutil.rmtree(REPO, ignore_errors=True)

failed = [k for k, v in R.items() if v is None]
print("analysis steps ok:", [k for k, v in R.items() if v is not None])
assert not failed, f"analysis steps failed: {failed}"


## Limits worth remembering

- Kaggle has **no audio device** and no JUCE target, so nothing here measures
  `renderBlock` or a real driver callback. Use `NCBenchRender` on Windows for that.
- Absolute `ns_per_op` is machine-specific. Compare the same kernel between two
  Kaggle runs of the same machine class; the ratio column is a question, not a verdict.
- `resample.bank_build` and `export.measure_loudness_10s` are offline/message-thread
  paths; `spsc_queue.push_pop`, `resample.weights_lookup` and `tempo_map.*` are on the
  audio thread and have a hard budget.
- A green run here says the portable engine is correct and not slower. It says
  nothing about the app shell, the installer, or distribution readiness.
